In [9]:
import sys, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / 'cartpole.py').exists() else pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT))
import importlib, controllers, evaluate as ev
from controllers import *
import cartpole as cp
for mod in (cp, controllers, ev): importlib.reload(mod)


P = cp.CartPoleParams(); A, B = cp.analytic_AB(P); dt = 0.02; A_d, B_d = cp.discretize(A, B, dt)
q = np.diag([4.0, 1.0, 25.0, 1.0])
K_bry, P_bry = lqr_discrete(A_d, B_d, q, np.array([[0.01]]), dt)
K_r10, _     = lqr_discrete(A_d, B_d, q, np.array([[0.1]]),  dt)

def lqg_yap(sigma=0.005, sigma_w=1.0):                        # KARARLAR: σ_w = 1 (kestirici kontrolcüden hızlı)
    kf = KalmanFilter(A_d, B_d, cp.H_XTHETA, B_d @ B_d.T * sigma_w**2 + 1e-8 * np.eye(4), np.eye(2) * sigma**2,
                      s0=np.zeros(4), P0=np.diag([sigma**2, 1.0, sigma**2, 1.0]))
    return LQGController(K_bry, kf)

# (ad, üretici, sensör σ)
kontrolculer = [
    ("Kaskad PID",     lambda: CascadePID(),                                                   None),
    ("LQR Bryson",     lambda: LQRController(K_bry),                                           None),
    ("LQR R×10",       lambda: LQRController(K_r10),                                           None),
    ("MPC N=10",       lambda: MPCController(A_d, B_d, q, np.array([[0.01]]), P_bry, N=10, dt=dt, K_yedek=K_bry), None),
    ("SonluFark σ.005", lambda: FiniteDiffLQR(K_bry),                                          0.005),
    ("LQG σ.005",      lambda: lqg_yap(),                                                      0.005),
]

sonuclar = {}
for blok_adi in ("nominal", "zor"):
    blok = ev.senaryolar(blok_adi, 50, seed=0)                     # GELİŞTİRME bloğu
    satirlar = []
    for ad, yap, sigma in kontrolculer:
        satirlar += ev.degerlendir(ad, yap, blok, sensor_sigma=sigma)
    sonuclar[blok_adi] = satirlar
    print(f"\n=== {blok_adi.upper()} blok (geliştirme, 50 senaryo) ===")
    ev.ozet_yazdir(ev.ozet(satirlar))
    ev.csv_yaz(satirlar, f"sonuclar_{blok_adi}_gelistirme.csv")

print()
ev.eslestirilmis(sonuclar["zor"], "Kaskad PID", "LQR R×10", olcut="oturma")
ev.eslestirilmis(sonuclar["zor"], "Kaskad PID", "LQR R×10", olcut="efor")


=== NOMINAL blok (geliştirme, 50 senaryo) ===
kontrolcü        |  başarı |    %95 Wilson | oturma med | efor med | max|x|
Kaskad PID       |  50/50  | [0.93, 1.00] |     3.02 s |      5.7 |   0.71
LQR Bryson       |  50/50  | [0.93, 1.00] |     2.15 s |      2.9 |   0.66
LQR R×10         |  50/50  | [0.93, 1.00] |     2.31 s |      1.7 |   0.68
MPC N=10         |  50/50  | [0.93, 1.00] |     2.15 s |      2.9 |   0.66
SonluFark σ.005  |  21/50  | [0.29, 0.56] |    10.00 s |    567.8 |   0.67
LQG σ.005        |  50/50  | [0.93, 1.00] |     2.85 s |      9.8 |   0.66


c:\Users\USER\miniconda3\envs\GNC_env\lib\site-packages\cvxpy\problems\problem.py:1539: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(



=== ZOR blok (geliştirme, 50 senaryo) ===
kontrolcü        |  başarı |    %95 Wilson | oturma med | efor med | max|x|
Kaskad PID       |  45/50  | [0.79, 0.96] |     3.72 s |     38.6 |  10.84
LQR Bryson       |  42/50  | [0.71, 0.92] |     2.49 s |     19.0 | 389.64
LQR R×10         |  45/50  | [0.79, 0.96] |     2.80 s |     14.1 | 376.22
MPC N=10         |  43/50  | [0.74, 0.93] |     2.54 s |     19.0 | 389.13
SonluFark σ.005  |  15/50  | [0.19, 0.44] |     9.96 s |    577.0 | 388.53
LQG σ.005        |  42/50  | [0.71, 0.92] |     2.91 s |     23.1 | 389.99

Kaskad PID vs LQR R×10: ikisi de başarılı 45, sadece Kaskad PID 0, sadece LQR R×10 0
  oturma farkı (LQR R×10 − Kaskad PID): ort -1.021, %95 aralık [-1.171, -0.864], LQR R×10 daha küçük olduğu senaryo oranı 1.00
Kaskad PID vs LQR R×10: ikisi de başarılı 45, sadece Kaskad PID 0, sadece LQR R×10 0
  efor farkı (LQR R×10 − Kaskad PID): ort -12.853, %95 aralık [-17.544, -8.225], LQR R×10 daha küçük olduğu senaryo oranı 0.87


(array([-1.42169523e+01, -4.44713950e+01, -1.62394187e+01, -2.76211070e+00,
        -9.29908577e+00, -1.65101075e+00,  1.36112851e+00, -2.54928467e+01,
        -6.07508069e+00, -4.35775512e+01, -6.19292547e+00,  2.59584365e+01,
        -5.09059400e+00, -1.43557730e+01, -1.63520841e+00, -6.29937859e+00,
        -3.23333427e+01, -2.98066252e+01, -3.78502004e+01, -9.71790608e+00,
        -3.38789200e+00, -1.72722298e+00, -5.67743118e+00,  2.80306510e+01,
        -2.70027954e+01,  2.32933841e-01, -4.62782656e+00, -1.46339509e+01,
        -3.75709666e+01, -3.62405469e+01, -1.77766283e+01, -1.60724388e+01,
        -1.01444050e+01, -2.48407096e+01, -1.11650914e+01, -1.34807383e+01,
        -3.38677454e+01, -1.56586201e+01, -5.24340464e-02, -3.10687155e+01,
        -1.46039119e+01, -1.94205202e+01, -2.67331150e+00,  1.47723175e+01,
         1.68626512e-02]),
 (np.float64(-17.543586068960966), np.float64(-8.225273421643802)))